# BUSA 695 Project Data Preparation Notebook

This notebook performs the early preparation of the original realtor dataset for the BUSA 695 capstone project. It loads the raw file, explores its structure, fills or removes missing values, converts the property status field into model-ready columns, and saves the cleaned output used by later notebooks.

**Dataset loaded:** `realtor-data.csv`

**Main steps:**
- Load and inspect the original dataset.
- Keep a mapping copy with location fields.
- Review shape, columns, data types, summary statistics, and missing values.
- Fill missing values and remove records without a target price.
- Drop less useful columns and encode the `status` field.
- Save the cleaned project dataset.

**Outputs created:** `realtor_master.csv`

**Capstone connection:** This is the main dataset-preparation notebook that creates the cleaned master file used by the cleaning, modeling, mapping, and random forest notebooks.
            

In [137]:
#importing libraries 
import os 
import pandas as pd    
import csv


## Load and Inspect the Raw Dataset

This section opens the original realtor data file and reviews its size, columns, data types, and basic descriptive statistics.
            

In [138]:
# Import the libraries used to inspect and clean the raw project dataset.
#creating a file path to csv info 
file_path = os.path.join("CAPSTONE_PEOJECT_BUSA_695", "realtor-data.csv")

In [139]:
#creating a dataframe to explore the data 
realtor_data_df = pd.read_csv("realtor-data.csv")

realtor_data_df

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2226377,23009.0,sold,359900.0,4.0,2.0,0.33,353094.0,Richland,Washington,99354.0,3600.0,2022-03-25
2226378,18208.0,sold,350000.0,3.0,2.0,0.10,1062149.0,Richland,Washington,99354.0,1616.0,2022-03-25
2226379,76856.0,sold,440000.0,6.0,3.0,0.50,405677.0,Richland,Washington,99354.0,3200.0,2022-03-24
2226380,53618.0,sold,179900.0,2.0,1.0,0.09,761379.0,Richland,Washington,99354.0,933.0,2022-03-24


In [140]:
#dataset copy with all variables (city, state, and zip_code) for mappping before I drop these non-numnerical values
map_df = realtor_data_df.copy()

## Preserve Location Fields for Mapping

This step keeps a full-copy version of the raw dataset so location information remains available for the later mapping workflow.
            

In [141]:
#checking that city, state and zip_code are saved under map_df
all(col in map_df.columns for col in ["city", "state", "zip_code"])

#double check 
map_df[["city", "state", "zip_code"]]

,city,state,zip_code
0,Adjuntas,Puerto Rico,601.0
1,Adjuntas,Puerto Rico,601.0
2,Juana Diaz,Puerto Rico,795.0
3,Ponce,Puerto Rico,731.0
4,Mayaguez,Puerto Rico,680.0
...,...,...,...
2226377,Richland,Washington,99354.0
2226378,Richland,Washington,99354.0
2226379,Richland,Washington,99354.0
2226380,Richland,Washington,99354.0


In [142]:
#checking the size (rows, columns) in the realtor-data.csv database
realtor_data_df.shape

(2226382, 12)

In [143]:
#exploring the columns and their names in the csv file 
realtor_data_df.columns 

Index(['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'street',
       'city', 'state', 'zip_code', 'house_size', 'prev_sold_date'],
      dtype='object')

In [144]:
#column  category 
realtor_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226382 entries, 0 to 2226381
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   brokered_by     float64
 1   status          object 
 2   price           float64
 3   bed             float64
 4   bath            float64
 5   acre_lot        float64
 6   street          float64
 7   city            object 
 8   state           object 
 9   zip_code        float64
 10  house_size      float64
 11  prev_sold_date  object 
dtypes: float64(8), object(4)
memory usage: 203.8+ MB


In [145]:
realtor_data_df.describe()

,brokered_by,price,bed,bath,acre_lot,street,zip_code,house_size
count,2.221849e+06,2.224841e+06,1.745065e+06,1.714611e+06,1.900793e+06,2.215516e+06,2.226083e+06,1.657898e+06
mean,5.293989e+04,5.241955e+05,3.275841e+00,2.496440e+00,1.522303e+01,1.012325e+06,5.218668e+04,2.714471e+03
std,3.064275e+04,2.138893e+06,1.567274e+00,1.652573e+00,7.628238e+02,5.837635e+05,2.895408e+04,8.081635e+05
min,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.000000e+00
25%,2.386100e+04,1.650000e+05,3.000000e+00,2.000000e+00,1.500000e-01,5.063128e+05,2.961700e+04,1.300000e+03
50%,5.288400e+04,3.250000e+05,3.000000e+00,2.000000e+00,2.600000e-01,1.012766e+06,4.838200e+04,1.760000e+03
75%,7.918300e+04,5.500000e+05,4.000000e+00,3.000000e+00,9.800000e-01,1.521173e+06,7.807000e+04,2.413000e+03
max,1.101420e+05,2.147484e+09,4.730000e+02,8.300000e+02,1.000000e+05,2.001357e+06,9.999900e+04,1.040400e+09


In [146]:
# Fill missing values so the cleaned project dataset is more complete for downstream analysis.
realtor_data_df.head()

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,NaN


In [147]:
#Identifying null values in the dataset 
realtor_data_df.isnull().sum()

brokered_by         4533
status                 0
price               1541
bed               481317
bath              511771
acre_lot          325589
street             10866
city                1407
state                  8
zip_code             299
house_size        568484
prev_sold_date    734297
dtype: int64

## Handle Missing Values

These cells fill important feature gaps, remove records without a target price, and standardize missing location values.
            

In [148]:
#replacing all null values with the median to preserve data. 

realtor_data_df["bed"].fillna(realtor_data_df["bed"].median(), inplace = True)
realtor_data_df["bath"].fillna(realtor_data_df["bath"].median(), inplace = True)
realtor_data_df["acre_lot"].fillna(realtor_data_df["acre_lot"].median(), inplace = True)
realtor_data_df["house_size"].fillna(realtor_data_df["house_size"].median(), inplace = True)

C:\Users\A1990\AppData\Local\Temp\ipykernel_17868\378125397.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  realtor_data_df["bed"].fillna(realtor_data_df["bed"].median(), inplace = True)
C:\Users\A1990\AppData\Local\Temp\ipykernel_17868\378125397.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

In [149]:
realtor_data_df = realtor_data_df.dropna(subset = ["price"])

In [150]:
realtor_data_df["city"].fillna("unknown", inplace = True)
realtor_data_df["state"].fillna("unknown", inplace = True)
realtor_data_df["zip_code"].fillna("unknown", inplace = True)

C:\Users\A1990\AppData\Local\Temp\ipykernel_17868\1321459019.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  realtor_data_df["city"].fillna("unknown", inplace = True)
C:\Users\A1990\AppData\Local\Temp\ipykernel_17868\1321459019.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  realtor_data_df["city"].fillna("unknown", inplace = True)
C:\

In [151]:
#removing nonrelated columns for LM
realtor_data_df = realtor_data_df.drop(columns=["street"])

## Remove Unneeded Columns

This section drops columns that are not used in the main linear-model preparation workflow.
            

In [152]:
#removing nonrelated columns for LM 
realtor_data_df = realtor_data_df.drop(columns=["brokered_by"])


In [153]:
#removing nonrelated columns for LM 
realtor_data_df = realtor_data_df.drop(columns=["prev_sold_date"])

In [154]:
realtor_data_df.isnull().sum()

status        0
price         0
bed           0
bath          0
acre_lot      0
city          0
state         0
zip_code      0
house_size    0
dtype: int64

In [155]:
realtor_data_df.dtypes

status         object
price         float64
bed           float64
bath          float64
acre_lot      float64
city           object
state          object
zip_code       object
house_size    float64
dtype: object

In [156]:
realtor_data_df.dtypes

status         object
price         float64
bed           float64
bath          float64
acre_lot      float64
city           object
state          object
zip_code       object
house_size    float64
dtype: object

In [157]:
# Save the cleaned master dataset that the later capstone notebooks depend on.
realtor_data_df["status"].unique()

array(['for_sale', 'ready_to_build', 'sold'], dtype=object)

## Encode Property Status and Save the Cleaned Dataset

The final preparation step converts the `status` field into model-ready indicator columns and writes the cleaned dataset to disk.
            

In [158]:
realtor_data_df["status"] = realtor_data_df["status"].str.lower().str.strip()
realtor_data_df

,status,price,bed,bath,acre_lot,city,state,zip_code,house_size
0,for_sale,105000.0,3.0,2.0,0.12,Adjuntas,Puerto Rico,601.0,920.0
1,for_sale,80000.0,4.0,2.0,0.08,Adjuntas,Puerto Rico,601.0,1527.0
2,for_sale,67000.0,2.0,1.0,0.15,Juana Diaz,Puerto Rico,795.0,748.0
3,for_sale,145000.0,4.0,2.0,0.10,Ponce,Puerto Rico,731.0,1800.0
4,for_sale,65000.0,6.0,2.0,0.05,Mayaguez,Puerto Rico,680.0,1760.0
...,...,...,...,...,...,...,...,...,...
2226377,sold,359900.0,4.0,2.0,0.33,Richland,Washington,99354.0,3600.0
2226378,sold,350000.0,3.0,2.0,0.10,Richland,Washington,99354.0,1616.0
2226379,sold,440000.0,6.0,3.0,0.50,Richland,Washington,99354.0,3200.0
2226380,sold,179900.0,2.0,1.0,0.09,Richland,Washington,99354.0,933.0


In [159]:
realtor_data_df = pd.get_dummies(realtor_data_df, columns=["status"])

In [160]:
realtor_data_df.columns 

Index(['price', 'bed', 'bath', 'acre_lot', 'city', 'state', 'zip_code',
       'house_size', 'status_for_sale', 'status_ready_to_build',
       'status_sold'],
      dtype='object')

In [161]:
cleaned_df = realtor_data_df.copy()

cleaned_df.to_csv("realtor_master.csv", index=False)